# VAZHI SFT v6.0 — Tamil Task Training on DAPT v2.1

**Key change from v5.x lineage:** Base model is DAPT v2.1 (Tamil-adapted vanilla) instead of vanilla/v5.1a. New lineage = new major version.

```
Lineage: Qwen3-0.6B → DAPT v2.1 (39.5M Tamil tokens) → SFT v6.0 (this notebook)
```

**DAPT v2.1 baseline:**
- Tamil word: 2% → 56% (+54%) — model now produces Tamil tokens
- Instruction following: 9/9 preserved
- English: still coherent
- But outputs are still gibberish — DAPT teaches language patterns, SFT teaches correct content

**SFT v6.0 goal:** Teach the DAPT'd model to give structured, correct Tamil responses.

**Dataset:** `CryptoYogi/vazhi-tamil-sft-v5_3` (4,264 samples: 3,837 train + 427 eval)
- Proven dataset from v5.3 — ChatML format with prompt/completion columns
- TRL v0.20+ native completion-only masking (no custom DataCollator needed)

**Runtime:** Colab Pro GPU (T4/L4/A100). ~30 min on A100, ~90 min on T4.

In [ ]:
# Cell 1 — Dependencies
!pip install -q -U \
  "transformers>=4.45.0,<5.0.0" \
  "trl>=0.20.0" \
  "datasets>=2.21.0" \
  "peft>=0.13.0" \
  "accelerate>=0.34.0" \
  "huggingface_hub>=0.24.7"

import torch
print(f"\u2705 Dependencies installed")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"   VRAM: {vram / 1024**3:.0f} GB")

In [ ]:
# Cell 2 — Configuration

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import re
import random
import glob
import gc
import shutil
import torch
import numpy as np
from collections import Counter, defaultdict
from datasets import load_dataset
from huggingface_hub import login, HfApi

from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainerCallback, LogitsProcessorList,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# === ENVIRONMENT DETECTION ===
IS_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR = "/kaggle/working" if IS_KAGGLE else "/content"
ENV_NAME = "Kaggle" if IS_KAGGLE else "Colab"

# === KEY CONFIG ===
BASE_MODEL = "CryptoYogi/vazhi-dapt-v2_1"           # DAPT v2.1 (Tamil-adapted vanilla Qwen3-0.6B)
VANILLA_MODEL = "Qwen/Qwen3-0.6B"                   # Vanilla baseline for comparison
SFT_DATASET = "CryptoYogi/vazhi-tamil-sft-v5_3"     # Same proven dataset
OUTPUT_MODEL = "CryptoYogi/vazhi-v6_0"               # Final VAZHI model
ADAPTER_REPO = "CryptoYogi/vazhi-v6_0-lora"          # Adapter backup

# Training config
LEARNING_RATE = 1e-5       # Conservative (proven in v5.0/v5.1a/v5.3)
NUM_EPOCHS = 1             # Start with 1, eval before deciding on epoch 2
MAX_LENGTH = 2048          # TRL v0.20+ max_length
LORA_R = 16                # Proven in all successful runs
LORA_ALPHA = 32            # Standard 2x ratio
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
BATCH_SIZE = 4             # Per-device (adjust for GPU: 4=T4, 8=L4, 16=A100)
GRADIENT_ACCUMULATION = 2  # Effective batch = BATCH_SIZE * n_gpu * GRADIENT_ACCUMULATION

# Qwen3 instruct <think> tokens to suppress during generation
THINK_TOKEN_IDS = [151667, 151668]

SYSTEM_PROMPT = (
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0bb5\u0bb4\u0bbf (VAZHI), "
    "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bc1 \u0bae\u0b95\u0bcd\u0b95\u0bb3\u0bc1\u0b95\u0bcd\u0b95\u0bbe\u0ba9 "
    "AI \u0b89\u0ba4\u0bb5\u0bbf\u0baf\u0bbe\u0bb3\u0bb0\u0bcd. "
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0ba4\u0bae\u0bbf\u0bb4\u0bbf\u0bb2\u0bcd \u0baa\u0ba4\u0bbf\u0bb2\u0bb3\u0bbf\u0baa\u0bcd\u0baa\u0bc0\u0bb0\u0bcd\u0b95\u0bb3\u0bcd."
)

# GPU auto-detection
assert torch.cuda.is_available(), "GPU required! Runtime > Change runtime type > GPU"
gpu_name = torch.cuda.get_device_name(0).lower()
_props = torch.cuda.get_device_properties(0)
VRAM_GB = getattr(_props, 'total_memory', getattr(_props, 'total_mem', 0)) / 1e9
IS_HIGH_END_GPU = any(x in gpu_name for x in ["a100", "l4", "h100", "a10"])
USE_BF16 = IS_HIGH_END_GPU
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
n_gpus = torch.cuda.device_count()
effective_batch = BATCH_SIZE * n_gpus * GRADIENT_ACCUMULATION

print(f"\u2705 SFT v6.0 Configuration:")
print(f"   Environment: {ENV_NAME}")
print(f"   Base model:  {BASE_MODEL} (DAPT v2.1)")
print(f"   Dataset:     {SFT_DATASET}")
print(f"   Output:      {OUTPUT_MODEL}")
print(f"   LR:          {LEARNING_RATE}")
print(f"   LoRA:        r={LORA_R}, alpha={LORA_ALPHA}, {len(LORA_TARGETS)} modules")
print(f"   Batch:       {BATCH_SIZE} x {GRADIENT_ACCUMULATION} accum = {effective_batch} effective")
print(f"   GPU:         {torch.cuda.get_device_name(0)} ({VRAM_GB:.0f} GB)")
print(f"   Precision:   {'bf16' if USE_BF16 else 'fp16'}")

In [ ]:
# Cell 3 — HuggingFace Login
from huggingface_hub import notebook_login
notebook_login()
print("\u2705 Logged in to HuggingFace")

In [ ]:
# Cell 4 — Helper Functions
#
# Tamil WORD validation (not just char %) — catches transliterated English gibberish.
# SuppressThinkTokens — prevents <think> leakage in generation.
# build_chat_prompt — constructs ChatML prompt with VAZHI system prompt.

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.padding_side = "right"

# Verify ChatML tokens
for token in ["<|im_start|>", "<|im_end|>"]:
    assert token in tokenizer.get_vocab(), f"Missing {token}!"
print(f"\u2705 Tokenizer: {len(tokenizer)} tokens, ChatML OK")

# Check enable_thinking support
try:
    tokenizer.apply_chat_template(
        [{"role": "user", "content": "test"}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    USE_THINKING_FLAG = True
except TypeError:
    USE_THINKING_FLAG = False


class SuppressThinkTokens:
    """Suppress specific token IDs by setting their logits to -inf."""
    def __init__(self, token_ids, device):
        self.suppress_ids = torch.tensor(token_ids, dtype=torch.long, device=device)

    def __call__(self, input_ids, scores):
        scores[:, self.suppress_ids] = float('-inf')
        return scores


def build_chat_prompt(user_text):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    if USE_THINKING_FLAG:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


def strip_think_tags(text):
    text = re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL)
    text = re.sub(r'</?think>', '', text)
    return text.strip()


def extract_response(full_text):
    if "<|im_start|>assistant" in full_text:
        resp = full_text.split("<|im_start|>assistant")[-1]
        resp = resp.split("<|im_end|>")[0].strip()
        if resp.startswith("\n"):
            resp = resp[1:]
    else:
        resp = full_text.strip()
    resp = strip_think_tags(resp)
    return resp


def tamil_char_pct(text):
    if not text:
        return 0.0
    total = sum(1 for c in text if not c.isspace() and not c.isdigit())
    if total == 0:
        return 0.0
    tamil = sum(1 for c in text if '\u0B80' <= c <= '\u0BFF')
    return 100.0 * tamil / total


def tamil_word_score(text):
    """Score based on Tamil word validation (not just char %)."""
    words = text.split()
    if not words:
        return 0.0, 0, 0
    tamil_words = 0
    for w in words:
        clean = re.sub(r'[\d\W]', '', w)
        if not clean:
            continue
        tamil_chars = sum(1 for c in clean if '\u0B80' <= c <= '\u0BFF')
        if tamil_chars / len(clean) > 0.5:
            tamil_words += 1
    return 100.0 * tamil_words / len(words), tamil_words, len(words)


def compute_repeat_ratio(text, n=3):
    """Detect repetitive output via n-gram ratio."""
    words = text.split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    if not ngrams:
        return 0.0
    return 1.0 - len(set(ngrams)) / len(ngrams)


def generate_response(model, prompt_text, max_new_tokens=200):
    """Generate a response using ChatML format."""
    full_prompt = build_chat_prompt(prompt_text)
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    
    if hasattr(model, 'generation_config') and hasattr(model.generation_config, 'suppress_tokens'):
        model.generation_config.suppress_tokens = None
    
    suppressor = SuppressThinkTokens(THINK_TOKEN_IDS, model.device)
    procs = LogitsProcessorList([suppressor])
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            logits_processor=procs,
        )
    return extract_response(tokenizer.decode(out[0], skip_special_tokens=False))


print("\u2705 Helpers ready")

In [ ]:
# Cell 5 — Pre-SFT Baseline on DAPT v2.1 Model
#
# Record DAPT model outputs BEFORE SFT for comparison.
# The DAPT model should produce Tamil tokens but incoherent content.
# SFT should make responses structured and correct.

print("\U0001f4ca Pre-SFT Baseline: DAPT v2.1 Model Outputs")
print("=" * 60)

BASELINE_PROMPTS = [
    ("\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "greeting"),
    ("\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "identity"),
    ("\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf", "thanks"),
    ("\u0b95\u0bbe\u0bb2\u0bc8\u0baf\u0bbf\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bbe\u0baa\u0bcd\u0baa\u0bbf\u0b9f\u0bb2\u0bbe\u0bae\u0bcd?", "health"),
    ("\u0bb0\u0bc7\u0bb7\u0ba9\u0bcd \u0b95\u0bbe\u0bb0\u0bcd\u0b9f\u0bc1 \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0ba4\u0b95\u0bb5\u0bb2\u0bcd \u0ba4\u0bc7\u0bb5\u0bc8", "govt"),
    ("\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "govt"),
    ("\u0ba8\u0bc0\u0bb0\u0bbf\u0bb4\u0bbf\u0bb5\u0bc1 \u0ba8\u0bcb\u0baf\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "health"),
    ("\u0b92\u0bb0\u0bc1 \u0ba4\u0bc6\u0bb0\u0bbf\u0baf\u0bbe\u0ba4 \u0b8e\u0ba3\u0bcd\u0ba3\u0bbf\u0bb2\u0bcd \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0bae\u0bc6\u0b9a\u0bc7\u0b9c\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0ba4\u0bc1", "safety"),
    ("\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bc1\u0bb1\u0bb3\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "culture"),
    ("\u0b95\u0bbe\u0baf\u0bcd\u0b9a\u0bcd\u0b9a\u0bb2\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?", "health"),
]

# Load DAPT model for baseline
print(f"\n\U0001f4e5 Loading {BASE_MODEL} for baseline...")
baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=MODEL_DTYPE, device_map={"": 0}, trust_remote_code=True,
)
baseline_model.eval()
baseline_model.config.use_cache = True

pre_sft_results = []
for prompt_text, category in BASELINE_PROMPTS:
    resp = generate_response(baseline_model, prompt_text)
    t_pct = tamil_char_pct(resp)
    tw_pct, tw_count, tw_total = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    pre_sft_results.append({
        'prompt': prompt_text, 'category': category,
        'response': resp[:300], 'tamil_char_pct': t_pct,
        'tamil_word_pct': tw_pct, 'repeat_ratio': rep,
    })
    print(f"\n[{category}] Char: {t_pct:.0f}%, Word: {tw_pct:.0f}%, Rep: {rep:.2f}")
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")

avg_pre_char = np.mean([r['tamil_char_pct'] for r in pre_sft_results])
avg_pre_word = np.mean([r['tamil_word_pct'] for r in pre_sft_results])
avg_pre_rep = np.mean([r['repeat_ratio'] for r in pre_sft_results])
print(f"\n\U0001f4ca DAPT v2.1 baseline: char {avg_pre_char:.0f}%, word {avg_pre_word:.0f}%, rep {avg_pre_rep:.2f}")

del baseline_model
gc.collect(); torch.cuda.empty_cache()
print("\U0001f5d1\ufe0f Baseline model freed")

In [ ]:
# Cell 6 — Load & Validate Dataset v5.3

print(f"\U0001f4da Loading SFT dataset from {SFT_DATASET}...")

try:
    sft_ds = load_dataset(SFT_DATASET)
    if "train" in sft_ds and "validation" in sft_ds:
        train_ds = sft_ds["train"]
        eval_ds = sft_ds["validation"]
    else:
        raise KeyError("No train/validation split")
except (KeyError, ValueError):
    train_ds = load_dataset("json", data_files={
        "train": f"hf://datasets/{SFT_DATASET}/vazhi-tamil-sft-v5_3-train.json"
    })["train"]
    eval_ds = load_dataset("json", data_files={
        "eval": f"hf://datasets/{SFT_DATASET}/vazhi-tamil-sft-v5_3-eval.json"
    })["eval"]

print(f"\u2705 Dataset loaded:")
print(f"   Train:      {len(train_ds)} samples")
print(f"   Validation: {len(eval_ds)} samples")
print(f"   Columns:    {train_ds.column_names}")

# Composition stats
if 'bucket' in train_ds.column_names:
    bucket_dist = Counter(item.get('bucket', 'unknown') for item in train_ds)
    print(f"\n\U0001f4ca Composition:")
    for bucket, count in sorted(bucket_dist.items(), key=lambda x: -x[1]):
        print(f"   {bucket}: {count} ({100*count/len(train_ds):.1f}%)")

# ChatML validation
CHATML_RE = re.compile(
    r'<\|im_start\|>system\n.+?<\|im_end\|>\n'
    r'<\|im_start\|>user\n(.+?)<\|im_end\|>\n'
    r'<\|im_start\|>assistant\n(.+?)<\|im_end\|>',
    re.DOTALL
)

# Check if dataset has 'text' column (full ChatML) or prompt/completion
has_text = 'text' in train_ds.column_names
has_prompt = 'prompt' in train_ds.column_names and 'completion' in train_ds.column_names

if has_text:
    fail_count = 0
    for i in range(len(train_ds)):
        if not CHATML_RE.search(train_ds[i]["text"]):
            fail_count += 1
            if fail_count <= 3:
                print(f"   \u274c Sample {i}: invalid ChatML")
    if fail_count == 0:
        print(f"\n\u2705 All {len(train_ds)} train samples pass ChatML validation")
    else:
        print(f"\n\u26a0\ufe0f {fail_count} samples failed ChatML validation")
elif has_prompt:
    print(f"\n\u2705 Dataset uses prompt/completion format (TRL native masking)")
    # Verify non-empty completions
    empty = sum(1 for item in train_ds if len(item['completion'].strip()) < 5)
    print(f"   Empty completions: {empty}")
else:
    raise ValueError(f"Dataset has unexpected columns: {train_ds.column_names}")

# Sample
print(f"\n\U0001f50d Sample prompt (first 200 chars):")
if has_prompt:
    print(f"   {train_ds[0]['prompt'][:200]}")
    print(f"\n\U0001f50d Sample completion (first 200 chars):")
    print(f"   {train_ds[0]['completion'][:200]}")
else:
    print(f"   {train_ds[0]['text'][:400]}")

In [ ]:
# Cell 7 — Load Model + LoRA Setup

print(f"\U0001f4e5 Loading {BASE_MODEL} (DAPT v2.1)...")

# NO device_map for training — prevents DataParallel issues
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=MODEL_DTYPE,
    trust_remote_code=True,
)
model = model.to("cuda:0")

model.config.pad_token_id = tokenizer.eos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable()

has_device_map = hasattr(model, "hf_device_map")
print(f"   hf_device_map present: {has_device_map} (must be False)")

mem_gb = torch.cuda.memory_allocated(0) / 1024**3
print(f"\u2705 Model loaded: {model.num_parameters():,} params | GPU: {mem_gb:.1f} GB")

# Apply LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

mem_gb = torch.cuda.memory_allocated(0) / 1024**3
print(f"\u2705 LoRA applied | GPU: {mem_gb:.1f} GB")

In [ ]:
# Cell 8 — Dataset Preflight: Token Length Validation

print(f"\U0001f4ca Token length distribution (train):")
token_lengths = []
if has_prompt:
    for idx in range(len(train_ds)):
        p_len = len(tokenizer.encode(train_ds[idx]["prompt"], add_special_tokens=False))
        c_len = len(tokenizer.encode(train_ds[idx]["completion"], add_special_tokens=False))
        token_lengths.append(p_len + c_len)
else:
    for idx in range(len(train_ds)):
        token_lengths.append(len(tokenizer.encode(train_ds[idx]["text"], add_special_tokens=False)))

token_lengths = np.array(token_lengths)
truncated = (token_lengths > MAX_LENGTH).sum()

print(f"   Mean: {token_lengths.mean():.0f}, Max: {token_lengths.max()}, P95: {np.percentile(token_lengths, 95):.0f}")
print(f"   Truncated (>{MAX_LENGTH}): {truncated} ({100*truncated/len(train_ds):.1f}%)")

if truncated > len(train_ds) * 0.1:
    print(f"   \u26a0\ufe0f >10% truncated — consider increasing MAX_LENGTH")
else:
    print(f"   \u2705 Token lengths OK")

In [ ]:
# Cell 9 — Training Setup

OUTPUT_DIR = f"{WORK_DIR}/sft-v6_0"

steps_per_epoch = len(train_ds) // effective_batch
total_steps = steps_per_epoch * NUM_EPOCHS
log_steps = max(total_steps // 30, 5)
eval_steps = max(steps_per_epoch // 3, 10)
save_steps = max(steps_per_epoch, 20)

print(f"\U0001f4ca Training Plan:")
print(f"   Train samples:    {len(train_ds)}")
print(f"   Eval samples:     {len(eval_ds)}")
print(f"   Effective batch:  {effective_batch}")
print(f"   Steps/epoch:      ~{steps_per_epoch}")
print(f"   Total steps:      ~{total_steps}")
print(f"   Eval every:       {eval_steps} steps")
print(f"   LoRA:             r={LORA_R}, {len(LORA_TARGETS)} modules")


class LossLoggingCallback(TrainerCallback):
    def __init__(self):
        self.losses = []
        self.eval_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if "loss" in logs:
                step = state.global_step
                loss = logs["loss"]
                lr = logs.get("learning_rate", 0)
                self.losses.append((step, loss))
                print(f"  Step {step:4d}/{total_steps} | Loss: {loss:.4f} | LR: {lr:.2e}")
            if "eval_loss" in logs:
                self.eval_losses.append((state.global_step, logs["eval_loss"]))
                print(f"  \U0001f4ca Eval Loss: {logs['eval_loss']:.4f}")


class MidTrainingGenCheck(TrainerCallback):
    """Generate Tamil responses mid-training to catch gibberish early."""

    SANITY_PROMPTS = [
        {"prompt": "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "label": "greeting"},
        {"prompt": "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "label": "identity"},
        {"prompt": "\u0b8e\u0ba9\u0b95\u0bcd\u0b95\u0bc1 \u0b89\u0ba4\u0bb5\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "label": "help"},
    ]

    def __init__(self, model_ref):
        self.model_ref = model_ref
        self.check_interval = max(steps_per_epoch // 2, 20)

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.check_interval != 0 or state.global_step == 0:
            return
        print(f"\n  \U0001f50d Mid-training gen check (step {state.global_step}):")
        self.model_ref.eval()
        self.model_ref.config.use_cache = True
        if hasattr(self.model_ref, 'gradient_checkpointing_disable'):
            self.model_ref.gradient_checkpointing_disable()

        for item in self.SANITY_PROMPTS:
            try:
                resp = generate_response(self.model_ref, item['prompt'], max_new_tokens=80)
                tw_pct, _, _ = tamil_word_score(resp)
                rep = compute_repeat_ratio(resp)
                print(f"    [{item['label']}] Word: {tw_pct:.0f}%, Rep: {rep:.2f} | {resp[:100]}")
            except Exception as e:
                print(f"    [{item['label']}] ERROR: {e}")

        self.model_ref.train()
        self.model_ref.config.use_cache = False
        if hasattr(self.model_ref, 'gradient_checkpointing_enable'):
            self.model_ref.gradient_checkpointing_enable()


loss_cb = LossLoggingCallback()
gen_cb = MidTrainingGenCheck(model)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=log_steps,
    save_steps=save_steps,
    eval_steps=eval_steps,
    eval_strategy="steps",
    save_total_limit=3,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=1.0,
    optim="adamw_torch",
    report_to="none",
    seed=RANDOM_SEED,
    load_best_model_at_end=False,
    dataloader_pin_memory=True,
    max_length=MAX_LENGTH,
    packing=False,
    push_to_hub=True,
    hub_model_id=ADAPTER_REPO,
    hub_strategy="every_save",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
    processing_class=tokenizer,
    callbacks=[loss_cb, gen_cb],
)

print(f"\u2705 SFTTrainer ready")
print(f"   Base: {BASE_MODEL} (DAPT v2.1 — Tamil-adapted)")
print(f"   LR: {LEARNING_RATE}, Epochs: {NUM_EPOCHS}")
print(f"   LoRA: r={LORA_R}, {len(LORA_TARGETS)} modules")

In [ ]:
# Cell 10 — Run Training

print(f"\U0001f680 Starting SFT v6.0 training...")
print(f"   ~{total_steps} steps, {NUM_EPOCHS} epoch")
print(f"   Base: {BASE_MODEL} (DAPT v2.1)")
print(f"   LR: {LEARNING_RATE}, Dataset: {len(train_ds)} train")
print()

train_result = trainer.train()

print("\n\u2705 Training complete!")
metrics = train_result.metrics
for k, v in metrics.items():
    print(f"   {k}: {v}")

print("\n\U0001f4ca Final eval...")
eval_metrics = trainer.evaluate()
for k, v in eval_metrics.items():
    print(f"   {k}: {v}")

if loss_cb.losses:
    s = loss_cb.losses[0][1]
    e = loss_cb.losses[-1][1]
    print(f"\n\U0001f4c8 Loss: {s:.4f} \u2192 {e:.4f} ({100*(s-e)/s:.1f}% drop)")

trainer.save_model()
trainer.push_to_hub()

In [ ]:
# Cell 10r — Resume Training (if Colab disconnected)
# Uncomment and run ONLY if Cell 10 was interrupted

# print("Resuming training from checkpoint...")
# train_result = trainer.train(resume_from_checkpoint=True)
# trainer.save_model()
# trainer.push_to_hub()

print("Cell 10r: Resume cell (commented out). Uncomment only if needed.")

In [ ]:
# Cell 11 — Save Adapter + A/B Test + Merge
#
# Test BOTH adapter inference and merged inference.
# If adapter works but merged doesn't, the merge is the problem.

ADAPTER_PATH = f"{WORK_DIR}/vazhi-sft-v6_0-lora"

print("\U0001f4be Saving LoRA adapter...")
trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"\u2705 Adapter saved to {ADAPTER_PATH}")

# Upload adapter backup
api = HfApi()
api.create_repo(ADAPTER_REPO, exist_ok=True)
print(f"\U0001f4e4 Uploading adapter to {ADAPTER_REPO}...")
api.upload_folder(
    folder_path=ADAPTER_PATH,
    repo_id=ADAPTER_REPO,
    commit_message=f"SFT v6.0 adapter: DAPT v2.1 base, {len(train_ds)} samples, r={LORA_R}, lr={LEARNING_RATE}",
)
print(f"\u2705 Adapter uploaded")

# Free training model
del model, trainer
gc.collect(); torch.cuda.empty_cache()
print("\U0001f5d1\ufe0f Training model freed")

AB_PROMPTS = [
    "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd",
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?",
    "\u0b95\u0bbe\u0bb2\u0bc8\u0baf\u0bbf\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bbe\u0baa\u0bcd\u0baa\u0bbf\u0b9f\u0bb2\u0bbe\u0bae\u0bcd?",
    "\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd",
    "\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf",
]

# --- Test A: Adapter inference ---
print("\n" + "=" * 60)
print("\U0001f1e6 TEST A: Adapter Inference (no merge)")
print("=" * 60)

base_a = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"":0}, trust_remote_code=True,
)
adapter_model = PeftModel.from_pretrained(base_a, ADAPTER_PATH)
adapter_model.eval()
adapter_model.config.use_cache = True

adapter_results = []
for prompt_text in AB_PROMPTS:
    resp = generate_response(adapter_model, prompt_text)
    tw_pct, _, _ = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    adapter_results.append({"resp": resp, "word": tw_pct, "rep": rep})
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")
    print(f"  (Word:{tw_pct:.0f}% Rep:{rep:.2f})")
    print()

del adapter_model, base_a
gc.collect(); torch.cuda.empty_cache()

# --- Test B: Merged model ---
print("\n" + "=" * 60)
print("\U0001f1e7 TEST B: Merged Model (fp16)")
print("=" * 60)

base_b = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"":0}, trust_remote_code=True,
)
peft_b = PeftModel.from_pretrained(base_b, ADAPTER_PATH)
peft_b.gradient_checkpointing_disable()
peft_b.config.use_cache = True
peft_b.eval()

print("\U0001f500 Merging LoRA in fp16...")
merged_model = peft_b.merge_and_unload()
if hasattr(merged_model, 'generation_config'):
    merged_model.generation_config.suppress_tokens = None

merged_results = []
for prompt_text in AB_PROMPTS:
    resp = generate_response(merged_model, prompt_text)
    tw_pct, _, _ = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    merged_results.append({"resp": resp, "word": tw_pct, "rep": rep})
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")
    print(f"  (Word:{tw_pct:.0f}% Rep:{rep:.2f})")
    print()

# A/B comparison
print("\n" + "=" * 60)
print("A/B COMPARISON")
print("=" * 60)
avg_a_word = np.mean([r['word'] for r in adapter_results])
avg_b_word = np.mean([r['word'] for r in merged_results])
print(f"   Adapter avg word: {avg_a_word:.0f}%")
print(f"   Merged avg word:  {avg_b_word:.0f}%")
if abs(avg_a_word - avg_b_word) > 15:
    print(f"   \u26a0\ufe0f MERGE CORRUPTION DETECTED — use adapter for deployment")
else:
    print(f"   \u2705 Merge OK — adapter and merged consistent")

In [ ]:
# Cell 12 — Full Eval: 16 Conversational Prompts
#
# Tamil WORD validation (not just char %) catches transliterated English gibberish.

merged_model.eval()
merged_model.config.use_cache = True

test_prompts = [
    {"prompt": "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "check": "greeting", "cat": "greeting"},
    {"prompt": "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "check": "identity", "cat": "greeting"},
    {"prompt": "\u0b8e\u0ba9\u0b95\u0bcd\u0b95\u0bc1 \u0b89\u0ba4\u0bb5\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "general", "cat": "help"},
    {"prompt": "\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf", "check": "general", "cat": "help"},
    {"prompt": "\u0ba8\u0bbe\u0ba9\u0bcd \u0b92\u0bb0\u0bc1 \u0baa\u0bbf\u0bb0\u0b9a\u0bcd\u0b9a\u0ba9\u0bc8 \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b95\u0bb5\u0bb2\u0bc8\u0baa\u0bcd\u0baa\u0b9f\u0bc1\u0b95\u0bbf\u0bb1\u0bc7\u0ba9\u0bcd. \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf\u0bb2\u0bbe\u0bae\u0bcd?", "check": "general", "cat": "help"},
    {"prompt": "\u0b92\u0bb0\u0bc1 \u0ba4\u0bc6\u0bb0\u0bbf\u0baf\u0bbe\u0ba4 \u0b8e\u0ba3\u0bcd\u0ba3\u0bbf\u0bb2\u0bcd \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0bae\u0bc6\u0b9a\u0bc7\u0b9c\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0ba4\u0bc1. \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0bb5\u0ba4\u0bc1?", "check": "safety", "cat": "safety"},
    {"prompt": "\u0bb5\u0bc0\u0b9f\u0bcd\u0b9f\u0bbf\u0bb2\u0bcd \u0ba4\u0bc0 \u0bb5\u0bbf\u0baa\u0ba4\u0bcd\u0ba4\u0bc1 \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?", "check": "safety", "cat": "safety"},
    {"prompt": "\u0ba8\u0bbe\u0bb3\u0bc8 \u0baa\u0b99\u0bcd\u0b95\u0bc1 \u0b9a\u0ba8\u0bcd\u0ba4\u0bc8 \u0b8f\u0bb1\u0bc1\u0bae\u0bbe?", "check": "refusal", "cat": "refusal"},
    {"prompt": "\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "govt"},
    {"prompt": "\u0bb0\u0bc7\u0bb7\u0ba9\u0bcd \u0b95\u0bbe\u0bb0\u0bcd\u0b9f\u0bc1 \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0ba4\u0b95\u0bb5\u0bb2\u0bcd \u0ba4\u0bc7\u0bb5\u0bc8", "check": "domain", "cat": "govt"},
    {"prompt": "\u0ba8\u0bc0\u0bb0\u0bbf\u0bb4\u0bbf\u0bb5\u0bc1 \u0ba8\u0bcb\u0baf\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "health"},
    {"prompt": "\u0b95\u0bbe\u0baf\u0bcd\u0b9a\u0bcd\u0b9a\u0bb2\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?", "check": "domain", "cat": "health"},
    {"prompt": "\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bc1\u0bb1\u0bb3\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "culture"},
    {"prompt": "FIR \u0baa\u0bcb\u0b9f\u0bc1\u0bb5\u0ba4\u0bc1 \u0b8e\u0baa\u0bcd\u0baa\u0b9f\u0bbf?", "check": "domain", "cat": "legal"},
    {"prompt": "\u0b95\u0bb2\u0bcd\u0bb5\u0bbf \u0b95\u0b9f\u0ba9\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0ba4\u0b95\u0bb5\u0bb2\u0bcd", "check": "domain", "cat": "education"},
    {"prompt": "\u0b9a\u0bc8\u0baa\u0bb0\u0bcd \u0bae\u0bcb\u0b9a\u0b9f\u0bbf\u0baf\u0bbf\u0bb2\u0bcd \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0baa\u0ba3\u0bae\u0bcd \u0b87\u0bb4\u0ba8\u0bcd\u0ba4\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf\u0bb2\u0bbe\u0bae\u0bcd?", "check": "safety", "cat": "security"},
]

print(f"{'='*70}")
print(f"\U0001f4ca FULL EVAL: 16 Conversational Prompts")
print(f"{'='*70}")

results = []
for item in test_prompts:
    resp = generate_response(merged_model, item['prompt'])
    t_pct = tamil_char_pct(resp)
    tw_pct, tw_count, tw_total = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    is_empty = len(resp.strip()) < 10
    
    results.append({
        'prompt': item['prompt'], 'cat': item['cat'],
        'response': resp[:300], 'tamil_char_pct': t_pct,
        'tamil_word_pct': tw_pct, 'repeat_ratio': rep,
        'is_empty': is_empty,
    })
    
    status = "\u274c EMPTY" if is_empty else ("\u26a0\ufe0f REP" if rep > 0.3 else "\u2705")
    print(f"\n[{item['cat']:>10}] {status} | Char: {t_pct:.0f}%, Word: {tw_pct:.0f}%, Rep: {rep:.2f}")
    print(f"  Q: {item['prompt']}")
    print(f"  A: {resp[:200]}")

# Summary
avg_char = np.mean([r['tamil_char_pct'] for r in results])
avg_word = np.mean([r['tamil_word_pct'] for r in results])
avg_rep = np.mean([r['repeat_ratio'] for r in results])
non_empty = sum(1 for r in results if not r['is_empty'])
high_rep = sum(1 for r in results if r['repeat_ratio'] > 0.3)

print(f"\n{'='*70}")
print(f"\U0001f4ca EVAL SUMMARY")
print(f"{'='*70}")
print(f"   Non-empty:      {non_empty}/{len(results)}")
print(f"   Avg Tamil char: {avg_char:.0f}%")
print(f"   Avg Tamil word: {avg_word:.0f}%")
print(f"   Avg repeat:     {avg_rep:.2f}")
print(f"   High repeat:    {high_rep}/{len(results)}")
print(f"")
print(f"   DAPT baseline:  char {avg_pre_char:.0f}%, word {avg_pre_word:.0f}%")
print(f"   Post-SFT:       char {avg_char:.0f}%, word {avg_word:.0f}%")
print(f"   \u0394 Char:         {avg_char - avg_pre_char:+.0f}%")
print(f"   \u0394 Word:         {avg_word - avg_pre_word:+.0f}%")

# GO / NO-GO
EVAL_PASSED = (non_empty >= 12 and avg_word >= 50 and high_rep <= 3)

if EVAL_PASSED:
    print(f"\n   \u2705 EVAL PASSED — proceed to upload")
else:
    print(f"\n   \u274c EVAL FAILED")
    if non_empty < 12:
        print(f"     Too many empty responses ({len(results) - non_empty})")
    if avg_word < 50:
        print(f"     Tamil word % too low ({avg_word:.0f}%)")
    if high_rep > 3:
        print(f"     Too many repetitive outputs ({high_rep})")

In [ ]:
# Cell 13 — Upload Merged Model

if not EVAL_PASSED:
    print("\u274c Eval did not pass. Skipping upload.")
    print(f"   Adapter available at: {ADAPTER_REPO}")
else:
    MERGED_PATH = f"{WORK_DIR}/vazhi-v6_0-merged"
    
    print(f"\U0001f4be Saving merged model to {MERGED_PATH}...")
    merged_model.save_pretrained(MERGED_PATH)
    tokenizer.save_pretrained(MERGED_PATH)
    
    api = HfApi()
    api.create_repo(OUTPUT_MODEL, exist_ok=True)
    
    print(f"\U0001f4e4 Uploading merged model to {OUTPUT_MODEL}...")
    api.upload_folder(
        folder_path=MERGED_PATH,
        repo_id=OUTPUT_MODEL,
        commit_message=(
            f"SFT v6.0: VAZHI Tamil assistant | "
            f"base={BASE_MODEL} (DAPT v2.1) | "
            f"LoRA r={LORA_R} x {len(LORA_TARGETS)} modules | "
            f"lr={LEARNING_RATE} | {len(train_ds)} samples | "
            f"Tamil word: {avg_pre_word:.0f}% -> {avg_word:.0f}%"
        ),
    )
    
    print(f"\n\u2705 Merged model: https://huggingface.co/{OUTPUT_MODEL}")
    print(f"\u2705 Adapter:      https://huggingface.co/{ADAPTER_REPO}")

In [ ]:
# Cell 14 — Summary

print(f"{'='*65}")
print(f"\U0001f4cb SFT v6.0 TRAINING SUMMARY")
print(f"{'='*65}")
print(f"")
print(f"   Lineage:     Qwen3-0.6B \u2192 DAPT v2.1 (39.5M tokens) \u2192 SFT v6.0")
print(f"   Base model:  {BASE_MODEL}")
print(f"   Dataset:     {SFT_DATASET} ({len(train_ds)} train / {len(eval_ds)} eval)")
print(f"   Output:      {OUTPUT_MODEL}")
print(f"")
print(f"   Training:")
print(f"     LR:          {LEARNING_RATE}")
print(f"     LoRA:        r={LORA_R}, alpha={LORA_ALPHA}, {len(LORA_TARGETS)} modules")
print(f"     Epochs:      {NUM_EPOCHS}")
print(f"     Steps:       ~{total_steps}")
print(f"")
print(f"   Results:")
print(f"     Tamil char:  {avg_pre_char:.0f}% \u2192 {avg_char:.0f}% (\u0394 {avg_char - avg_pre_char:+.0f}%)")
print(f"     Tamil word:  {avg_pre_word:.0f}% \u2192 {avg_word:.0f}% (\u0394 {avg_word - avg_pre_word:+.0f}%)")
print(f"     Non-empty:   {non_empty}/{len(results)}")
print(f"     Avg repeat:  {avg_rep:.2f}")
print(f"     Eval passed: {'\u2705 YES' if EVAL_PASSED else '\u274c NO'}")
print(f"")
print(f"   \U0001f449 Next steps:")
print(f"      1. If eval passed: GGUF conversion (Q4_K_M) for mobile")
print(f"      2. If Tamil quality still poor: try epoch 2 or adjust dataset")
print(f"      3. Test on mobile with Flutter app")